# RAG QRT su Kaggle

Notebook che usa il pacchetto `qrt_rag` di questo repository.

**Impostazioni:** Accelerator GPU T4 x2 · Internet ON · Dataset con il PDF e Dataset con il parsing · secret `GRADIO_USER` e `GRADIO_PASS`.

**Lingua:** `LANG = "en"` per il regolamento in inglese (domande in inglese), `"it"` per la versione italiana. Ogni lingua ha il suo PDF e il suo parsing.

In [ ]:
# Pacchetto dal repository GitHub.
# Se il repository non e' ancora aggiornato: carica la cartella del progetto come Dataset Kaggle
# e usa invece:  !pip install -q "/kaggle/input/<nome-dataset>/rag-qrt[parsing,ui,eval]"
# Dopo un aggiornamento del codice su GitHub: aggiungi --force-reinstall --no-deps e riavvia il kernel.
!pip install -q "qrt-rag[parsing,ui,eval] @ git+https://github.com/mat126/rag-qrt-SolvencyII.git"

In [ ]:
import glob
from pathlib import Path

import torch

LANG = "en"   # "en" oppure "it": lingua del documento e delle domande

pdfs = glob.glob("/kaggle/input/**/*.pdf", recursive=True)
print("PDF trovati:", pdfs)
PDF_PATH = pdfs[0] if pdfs else None   # con piu' PDF collegati, scrivi il percorso a mano

WORK = Path("/kaggle/working")
PAGES_DIR = WORK / "pages"   # output del parsing, se lo rifai
INDEX_DIR = WORK / "index"   # chunk + embeddings

# Dataset con il parsing gia' fatto per questa lingua; None per rifare il parsing
PARSED_INPUT = None   # es. Path("/kaggle/input/datasets/mat126/parsing-rag-qrt-en")

BATCH = 50          # pagine per batch di parsing
TEST_RANGE = None   # es. (0, 60) per una prova; None per tutto il PDF
MAX_CHARS = 1500    # lunghezza massima dei chunk di testo/tabella

print("Lingua:", LANG, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nessuna")

In [ ]:
# Parsing con Docling (circa 25 minuti): solo se non c'e' gia' un parsing salvato.
# Alla fine: !cd /kaggle/working && zip -rq pages.zip pages  e salvalo come Dataset.
from qrt_rag.parsing import run_parsing

if PARSED_INPUT is None:
    run_parsing(PDF_PATH, PAGES_DIR, WORK, batch=BATCH, page_range=TEST_RANGE)
else:
    print("Parsing gia' disponibile in", PARSED_INPUT, "- salto il parsing")

In [ ]:
# Chunking strutturale. Controlla in fondo i "Prefissi degli allegati": per l'inglese
# il titolo abbreviato dovrebbe essere tipo "Annex II (individual undertakings)".
from qrt_rag.chunking import build_chunks, print_chunk_stats
from qrt_rag.parsing import load_pages

pages = load_pages(PARSED_INPUT or PAGES_DIR)
chunks = build_chunks(pages, max_chars=MAX_CHARS, lang=LANG)
print_chunk_stats(chunks, n_pages=len(pages))

In [ ]:
# Embeddings bge-m3 e salvataggio dell'indice (circa 1 minuto)
from qrt_rag.index import embed_chunks, load_embedder, save_index

emb_model = load_embedder()
E = embed_chunks(emb_model, chunks, batch_size=16)   # 8 se va in out-of-memory
save_index(INDEX_DIR, chunks, E)

In [ ]:
# Ricerca ibrida: prove con alcune domande del set di valutazione
from qrt_rag.evaluation import load_eval_set
from qrt_rag.retrieval import Retriever, load_reranker, show

retriever = Retriever(chunks, E, emb_model, reranker=load_reranker(), lang=LANG)
eval_cases = {c["id"]: c for c in load_eval_set(lang=LANG)}

for cid in ["c01", "n06", "g01"]:
    print("=" * 100, "\nDOMANDA:", eval_cases[cid]["q"])
    show(retriever.search(eval_cases[cid]["q"], k=5))

print("Posizione di R0540:", retriever.rank_of(eval_cases["n06"]["q"], "R0540",
                                               template="S.23.01", annex="II"))

In [ ]:
# LLM locale (Qwen2.5-7B-Instruct): il modello si carica solo la prima volta.
# Per cambiare prompt senza ricaricarlo: llm.system_prompt = NUOVO_PROMPT
from qrt_rag.generation import LocalLLM, ask

if "llm" not in globals():
    llm = LocalLLM(lang=LANG)

_ = ask(retriever, llm, eval_cases["c01"]["q"])
_ = ask(retriever, llm, eval_cases["n06"]["q"], annex="II")

In [ ]:
# Interfaccia Gradio con login (secret GRADIO_USER e GRADIO_PASS).
# Il link da aprire e' quello che finisce in .gradio.live
from kaggle_secrets import UserSecretsClient

from qrt_rag.lang import UI
from qrt_rag.ui import build_demo

try:
    _sec = UserSecretsClient()
    AUTH = (_sec.get_secret("GRADIO_USER").strip(), _sec.get_secret("GRADIO_PASS").strip())
except Exception as ex:
    raise RuntimeError(
        "Mancano i secret GRADIO_USER e GRADIO_PASS: aggiungili in Add-ons > Secrets "
        "e attiva l'interruttore per questo notebook."
    ) from ex

demo = build_demo(retriever, llm=globals().get("llm"), feedback_path=WORK / "feedback.jsonl")
demo.launch(share=True, auth=AUTH, auth_message=UI[LANG]["auth"])

In [ ]:
# Valutazione: recupero (con e senza reranker) e, se vuoi, risposte del modello
from qrt_rag.evaluation import eval_llm, load_feedback_cases, report_retrieval

cases = load_eval_set(lang=LANG) + load_feedback_cases()
df_rr, df_no = report_retrieval(retriever, cases, out_dir=WORK)

RUN_LLM_EVAL = False                           # lenta: 20-30 s a domanda
LLM_CACHE = globals().get("LLM_CACHE", {})     # LLM_CACHE.clear() per ricominciare
if RUN_LLM_EVAL:
    df_llm = eval_llm(cases,
                      lambda q: ask(retriever, llm, q, k=10, show_sources=False)[0],
                      cache=LLM_CACHE, out_dir=WORK)

In [ ]:
# Diagnostica per stadio su una domanda che fallisce
from qrt_rag.evaluation import diagnose

for cid in ["g01", "n11", "n07"]:
    diagnose(retriever, cases, cid)
    print("=" * 100)